In [ ]:
# filename: train_fraud_models.py
import os
import gc
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from typing import Dict, Tuple

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_recall_fscore_support
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Optional: XGBoost if installed (recommended for large tabular data)
try:
    from xgboost import XGBClassifier
    _has_xgb = True
except Exception:
    _has_xgb = False

# For plotting (optional)
import matplotlib.pyplot as plt
import seaborn as sns

# Make deterministic-ish
RANDOM_STATE = 42


In [ ]:
df = pd.read_csv("Fraud.csv")  # change path if needed
print("Loaded rows:", len(df))
df.head()

Loaded rows: 151799


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0.0,0.0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0.0,0.0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1.0,0.0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1.0,0.0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0.0,0.0


In [ ]:
print(df.info())
print(df.isnull().sum())
print("Fraud distribution:\n", df["isFraud"].value_counts(normalize=False))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151799 entries, 0 to 151798
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   step            151799 non-null  int64  
 1   type            151799 non-null  object 
 2   amount          151799 non-null  float64
 3   nameOrig        151799 non-null  object 
 4   oldbalanceOrg   151799 non-null  float64
 5   newbalanceOrig  151798 non-null  float64
 6   nameDest        151798 non-null  object 
 7   oldbalanceDest  151798 non-null  float64
 8   newbalanceDest  151798 non-null  float64
 9   isFraud         151798 non-null  float64
 10  isFlaggedFraud  151798 non-null  float64
dtypes: float64(7), int64(1), object(3)
memory usage: 12.7+ MB
None
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    1
nameDest          1
oldbalanceDest    1
newbalanceDest    1
isFraud           1
isFlaggedFraud    1
dty

In [ ]:
df["payerdebited"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
df["recievercredited"] = df["newbalanceDest"] - df["oldbalanceDest"]

In [ ]:
# payer and receiver type (first char)
df["payer_type"] = df["nameOrig"].str[0]  # mostly 'C'
df["reciever_type"] = df["nameDest"].str[0]  # 'M' or 'C'

In [ ]:
start = pd.to_datetime("2024-04-01")
df["datetime"] = start + pd.to_timedelta(df["step"], unit="h")
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["date"] = df["datetime"].dt.day

In [ ]:
df_model = df.copy()

In [ ]:
df_model["isFraud_business"] = df_model["isFraud"]
df_model["isFraud_label"] = np.where(df_model["amount"] > 200000, 1, df_model["isFraud_business"])

In [ ]:
drop_cols = [
    "step", "datetime", "isFlaggedFraud",
    "nameOrig", "nameDest",
    # dropping raw balances to avoid leakage if you choose to; comment out if you want to keep them
    "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"
]

In [ ]:
for c in drop_cols:
    if c in df_model.columns:
        df_model.drop(columns=c, inplace=True)

print("Columns used:", df_model.columns.tolist())


Columns used: ['type', 'amount', 'isFraud', 'payerdebited', 'recievercredited', 'payer_type', 'reciever_type', 'hour', 'day_of_week', 'date', 'isFraud_business', 'isFraud_label']


In [ ]:
df_model = df_model.dropna(subset=["isFraud", "payerdebited", "recievercredited"])

In [ ]:

# Ensure target column exists
target = "isFraud_label"

# Replace infinities and NaNs before type conversion
df_model = df_model.replace([np.inf, -np.inf], np.nan)
df_model[target] = df_model[target].fillna(0)  # assume missing labels are non-fraud (0)

# Convert safely
df_model[target] = df_model[target].astype(int)

# Separate features and target
X = df_model.drop(columns=[target, "isFraud_business"], errors="ignore")
y = df_model[target]

print("✅ Cleaned successfully!")
print("X shape:", X.shape)
print("Target distribution:\n", y.value_counts())

✅ Cleaned successfully!
X shape: (151798, 10)
Target distribution:
 isFraud_label
0    109417
1     42381
Name: count, dtype: int64


In [ ]:
# Cell 6 — Light cleaning & column types
# Convert categorical columns to strings
cat_cols = []
if "type" in X.columns:
    cat_cols.append("type")
if "payer_type" in X.columns:
    cat_cols.append("payer_type")
if "reciever_type" in X.columns:
    cat_cols.append("reciever_type")

# Ensure they are strings (so OneHotEncoder picks them up)
for c in cat_cols:
    X[c] = X[c].astype(str)

numeric_cols = [c for c in X.columns if c not in cat_cols]
print("Numeric cols:", numeric_cols)
print("Categorical cols:", cat_cols)

Numeric cols: ['amount', 'isFraud', 'payerdebited', 'recievercredited', 'hour', 'day_of_week', 'date']
Categorical cols: ['type', 'payer_type', 'reciever_type']


In [ ]:

# Cell 7 — Train/validation split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (121438, 10) Test shape: (30360, 10)


In [ ]:
# Cell 8 — Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, cat_cols)
    ],
    verbose_feature_names_out=False
)

In [ ]:
# Cell 9 — Models to train
models: Dict[str, Pipeline] = {}

# Random Forest (balanced)
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=16,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
models["random_forest"] = Pipeline([("preprocessor", preprocessor), ("clf", rf)])

# Logistic Regression (as baseline)
lr = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1)
models["logistic_regression"] = Pipeline([("preprocessor", preprocessor), ("clf", lr)])

# XGBoost if available (use scale_pos_weight to handle imbalance)
if _has_xgb:
    # compute scale_pos_weight
    pos = sum(y_train == 1)
    neg = sum(y_train == 0)
    scale_pos_weight = neg / max(pos, 1)
    xgb = XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
    models["xgboost"] = Pipeline([("preprocessor", preprocessor), ("clf", xgb)])
else:
    print("XGBoost not installed — skipping XGBoost model.")

In [ ]:
print(X.isna().sum()[X.isna().sum() > 0])

Series([], dtype: int64)


In [ ]:
# Cell 10 — Train models (careful: this can take time on full dataset)
results = {}
for name, pipe in models.items():
    print(f"\nTraining {name} ...")
    pipe.fit(X_train, y_train)
    # Evaluate on test set
    y_pred = pipe.predict(X_test)
    if hasattr(pipe, "predict_proba"):
        y_prob = pipe.predict_proba(X_test)[:, 1]
    else:
        # Pipeline's classifier exists at step -1
        clf = pipe.named_steps["clf"]
        if hasattr(clf, "predict_proba"):
            y_prob = clf.predict_proba(pipe.named_steps["preprocessor"].transform(X_test))[:, 1]
        else:
            y_prob = pipe.named_steps["clf"].decision_function(pipe.named_steps["preprocessor"].transform(X_test))
            # scale to 0..1 via minmax (not ideal) — but most classifiers we use have predict_proba
            y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min() + 1e-9)

    auc = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, digits=4, output_dict=True)
    print("ROC-AUC:", auc)
    print(pd.DataFrame(report).transpose())
    results[name] = {
        "auc": float(auc),
        "report": report,
        "model_obj": pipe
    }

    # Save each pipeline
    model_path = f"{name}_pipeline.joblib"
    joblib.dump(pipe, model_path)
    print("Saved:", model_path)

    # free some memory if needed
    gc.collect()



Training random_forest ...
ROC-AUC: 1.0
              precision  recall  f1-score  support
0                   1.0     1.0       1.0  21884.0
1                   1.0     1.0       1.0   8476.0
accuracy            1.0     1.0       1.0      1.0
macro avg           1.0     1.0       1.0  30360.0
weighted avg        1.0     1.0       1.0  30360.0
Saved: random_forest_pipeline.joblib

Training logistic_regression ...
ROC-AUC: 0.9999960967990388
              precision    recall  f1-score       support
0              1.000000  0.992369  0.996170  21884.000000
1              0.980678  1.000000  0.990245   8476.000000
accuracy       0.994499  0.994499  0.994499      0.994499
macro avg      0.990339  0.996184  0.993207  30360.000000
weighted avg   0.994606  0.994499  0.994516  30360.000000
Saved: logistic_regression_pipeline.joblib

Training xgboost ...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:26:17] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


ROC-AUC: 0.9999959188907076
              precision    recall  f1-score       support
0              0.999132  0.999223  0.999178  21884.000000
1              0.997994  0.997758  0.997876   8476.000000
accuracy       0.998814  0.998814  0.998814      0.998814
macro avg      0.998563  0.998491  0.998527  30360.000000
weighted avg   0.998814  0.998814  0.998814  30360.000000
Saved: xgboost_pipeline.joblib


In [ ]:
import joblib
import pandas as pd

# Load the trained XGBoost pipeline
model = joblib.load("xgboost_pipeline.joblib")

In [ ]:
sample_tx = {
    "type": "TRANSFER",
    "amount": 250500.00,
    "payerdebited": 250500.00,
    "recievercredited": 0.0,
    "payer_type": "C",
    "reciever_type": "C",
    "hour": 14,
    "day_of_week": 2,
    "date": 13
}

# Convert dict to DataFrame (XGBoost expects DataFrame / 2D array)
df_tx = pd.DataFrame([sample_tx])


In [ ]:
# ===== Step 0: Imports =====
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
import joblib

# ===== Step 1: Load your dataset =====
df = pd.read_csv("Fraud.csv")

# ===== Step 2: Feature Engineering =====
# Compute debited/credited amounts
df["payerdebited"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
df["recievercredited"] = df["newbalanceDest"] - df["oldbalanceDest"]

# Extract datetime features
df['datetime'] = pd.to_datetime('2024-04-01') + pd.to_timedelta(df['step'], unit='h')
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['date'] = df['datetime'].dt.day

# Payer/Receiver type
df["payer_type"] = df["nameOrig"].str[0]
df["reciever_type"] = df["nameDest"].str[0]

# Drop unnecessary columns
df_model = df.drop(columns=["step","datetime","isFlaggedFraud",
                            "nameOrig","nameDest",
                            "oldbalanceOrg","newbalanceOrig",
                            "oldbalanceDest","newbalanceDest"])

# Fill any NaN or inf
df_model = df_model.replace([np.inf, -np.inf], np.nan)
df_model = df_model.fillna(0)

# ===== Step 3: Define features & target =====
target = "isFraud"
numeric_cols = ['amount', 'payerdebited', 'recievercredited', 'hour', 'day_of_week', 'date']
categorical_cols = ['type', 'payer_type', 'reciever_type']

X = df_model[numeric_cols + categorical_cols]
y = df_model[target].astype(int)

# ===== Step 4: Build preprocessing + XGBoost pipeline =====
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(use_label_encoder=False, eval_metric="logloss"))
])

# ===== Step 5: Train model =====
xgb_pipeline.fit(X, y)
print("✅ XGBoost pipeline trained successfully!")

# Save the pipeline
joblib.dump(xgb_pipeline, "xgboost_pipeline_fraud.pkl")
print("✅ Pipeline saved as 'xgboost_pipeline_fraud.pkl'")

# ===== Step 6: Predict function for new transactions =====
def predict_transaction(tx: dict):
    df_tx = pd.DataFrame([tx])
    pred = xgb_pipeline.predict(df_tx)[0]
    prob = xgb_pipeline.predict_proba(df_tx)[0][1]
    return {"isFraud": int(pred), "fraud_probability": float(prob)}

# ===== Step 7: Test prediction =====
sample_tx = {
    "type": "TRANSFER",
    "amount": 250500.00,
    "payerdebited": 250500.00,
    "recievercredited": 0.0,
    "payer_type": "C",
    "reciever_type": "C",
    "hour": 14,
    "day_of_week": 2,
    "date": 13
}

result = predict_transaction(sample_tx)
print("Prediction:", result)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:31:10] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ XGBoost pipeline trained successfully!
✅ Pipeline saved as 'xgboost_pipeline_fraud.pkl'
Prediction: {'isFraud': 1, 'fraud_probability': 1.0}


In [ ]:
# Sample transactions
sample_txs = [
    {
        "type": "TRANSFER",
        "amount": 250500.00,
        "payerdebited": 250500.00,
        "recievercredited": 0.0,
        "payer_type": "C",
        "reciever_type": "C",
        "hour": 14,
        "day_of_week": 2,
        "date": 13
    },
    {
        "type": "PAYMENT",
        "amount": 5000.00,
        "payerdebited": 5000.00,
        "recievercredited": 5000.00,
        "payer_type": "C",
        "reciever_type": "M",
        "hour": 10,
        "day_of_week": 0,
        "date": 5
    },
    {
        "type": "CASH_OUT",
        "amount": 100000.00,
        "payerdebited": 100000.00,
        "recievercredited": 0.0,
        "payer_type": "C",
        "reciever_type": "C",
        "hour": 18,
        "day_of_week": 4,
        "date": 20
    },
    {
        "type": "TRANSFER",
        "amount": 15000.00,
        "payerdebited": 15000.00,
        "recievercredited": 15000.00,
        "payer_type": "C",
        "reciever_type": "C",
        "hour": 9,
        "day_of_week": 6,
        "date": 28
    }
]

# Convert to DataFrame
df_tx = pd.DataFrame(sample_txs)

# Predict using pipeline
predictions = xgb_pipeline.predict(df_tx)
probabilities = xgb_pipeline.predict_proba(df_tx)[:,1]

# Show results
for i, tx in enumerate(sample_txs):
    print(f"Transaction {i+1}:")
    print(f"  Amount: {tx['amount']}, Type: {tx['type']}")
    print(f"  Prediction (isFraud): {predictions[i]}, Fraud Probability: {probabilities[i]:.6f}\n")


Transaction 1:
  Amount: 250500.0, Type: TRANSFER
  Prediction (isFraud): 1, Fraud Probability: 1.000000

Transaction 2:
  Amount: 5000.0, Type: PAYMENT
  Prediction (isFraud): 0, Fraud Probability: 0.000000

Transaction 3:
  Amount: 100000.0, Type: CASH_OUT
  Prediction (isFraud): 0, Fraud Probability: 0.102062

Transaction 4:
  Amount: 15000.0, Type: TRANSFER
  Prediction (isFraud): 0, Fraud Probability: 0.000004



| Feature            | How it's calculated / source            |
| ------------------ | --------------------------------------- |
| `type`             | Directly from dataset `type`            |
| `amount`           | Directly from dataset `amount`          |
| `payerdebited`     | `oldbalanceOrg - newbalanceOrig`        |
| `recievercredited` | `newbalanceDest - oldbalanceDest`       |
| `payer_type`       | First letter of `nameOrig` (`C` or `M`) |
| `reciever_type`    | First letter of `nameDest` (`C` or `M`) |
| `hour`             | `step` → datetime → `.hour`             |
| `day_of_week`      | `step` → datetime → `.dayofweek`        |
| `date`             | `step` → datetime → `.day`              |
